# MedQuAD — EDA

47,457 medical QA pairs from 12 NIH websites, stored as one XML file per question, grouped into folders per source (e.g. `9_CDC_QA`). Each XML has a `Focus` (the disease/drug/entity), and one or more `QAPair`s, each with a `Question` (with a `qtype` attribute like "treatment", "symptoms") and an `Answer`.

**Known gap, not a bug:** 3 subsets had their answers removed for MedlinePlus copyright reasons (A.D.A.M. Medical Encyclopedia, MedlinePlus Drug info, MedlinePlus Herbal medicine/supplements) — this notebook surfaces exactly how many rows that affects rather than silently dropping them.


In [ ]:
import os

REPO_DIR = "MedQuAD"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/abachaa/MedQuAD.git

# Folders that hold QA XML files (skip anything that isn't a numbered source folder)
source_folders = sorted([d for d in os.listdir(REPO_DIR)
                          if os.path.isdir(os.path.join(REPO_DIR, d)) and d[0].isdigit()])
print(f"{len(source_folders)} source folders found")
source_folders

12 source folders found


['10_MPlus_ADAM_QA',
 '11_MPlusDrugs_QA',
 '12_MPlusHerbsSupplements_QA',
 '1_CancerGov_QA',
 '2_GARD_QA',
 '3_GHR_QA',
 '4_MPlus_Health_Topics_QA',
 '5_NIDDK_QA',
 '6_NINDS_QA',
 '7_SeniorHealth_QA',
 '8_NHLBI_QA_XML',
 '9_CDC_QA']

## 1. Diagnostic: confirm XML structure on one file
Print the raw tags of a single file before parsing everything, so a mismatch shows up immediately instead of silently.

In [ ]:
import xml.etree.ElementTree as ET
import glob

sample_folder = source_folders[0]
sample_files = glob.glob(os.path.join(REPO_DIR, sample_folder, "*.xml"))
print(f"Inspecting: {sample_files[0]}")

tree = ET.parse(sample_files[0])
root = tree.getroot()

def print_structure(elem, depth=0):
    print("  " * depth + f"<{elem.tag}> attrs={elem.attrib}")
    for child in list(elem)[:3]:  # only first 3 children per level to keep it short
        print_structure(child, depth + 1)

print_structure(root)

Inspecting: MedQuAD/10_MPlus_ADAM_QA/0002079.xml
<Document> attrs={'id': '0002079', 'source': 'ADAM', 'url': 'https://www.nlm.nih.gov/medlineplus/ency/article/001361.htm'}
  <Focus> attrs={}
  <FocusAnnotations> attrs={}
    <Category> attrs={}
  <QAPairs> attrs={}
    <QAPair> attrs={'pid': '1'}
      <Question> attrs={'qid': '0002079-1', 'qtype': 'information'}
      <Answer> attrs={}


## 2. Parse all XML files into a flat table
One row per (question, answer) pair, across every source folder.

In [ ]:
rows = []
parse_errors = []

for folder in source_folders:
    for filepath in glob.glob(os.path.join(REPO_DIR, folder, "*.xml")):
        try:
            tree = ET.parse(filepath)
            root = tree.getroot()
            focus_elem = root.find("Focus")
            focus = focus_elem.text if focus_elem is not None else None

            qapairs = root.find("QAPairs")
            if qapairs is None:
                continue

            for qa in qapairs.findall("QAPair"):
                q_elem = qa.find("Question")
                a_elem = qa.find("Answer")
                rows.append({
                    "source_folder": folder,
                    "file": os.path.basename(filepath),
                    "focus": focus,
                    "qtype": q_elem.get("qtype") if q_elem is not None else None,
                    "question": q_elem.text if q_elem is not None else None,
                    "answer": a_elem.text if a_elem is not None else None,
                })
        except ET.ParseError as e:
            parse_errors.append((filepath, str(e)))

import pandas as pd
df = pd.DataFrame(rows)
print(f"Parsed {len(df)} QA pairs across {len(source_folders)} folders")
print(f"Parse errors: {len(parse_errors)}")
df.head()

Parsed 47441 QA pairs across 12 folders
Parse errors: 0


,source_folder,file,focus,qtype,question,answer
0,10_MPlus_ADAM_QA,0002079.xml,Hyperimmunization,information,Do you have information about Hyperimmunization,None
1,10_MPlus_ADAM_QA,0001597.xml,First aid kit,information,Do you have information about First aid kit,None
2,10_MPlus_ADAM_QA,0001016.xml,Cranial mononeuropathy III - diabetic type,information,What is (are) Cranial mononeuropathy III - dia...,None
3,10_MPlus_ADAM_QA,0001016.xml,Cranial mononeuropathy III - diabetic type,causes,What causes Cranial mononeuropathy III - diabe...,None
4,10_MPlus_ADAM_QA,0001016.xml,Cranial mononeuropathy III - diabetic type,symptoms,What are the symptoms of Cranial mononeuropath...,None


## 3. Answer coverage
How many QA pairs actually have an answer vs. the copyright-removed gaps.

In [ ]:
df["has_answer"] = df["answer"].notna() & (df["answer"].str.strip().str.len() > 0)

print("Overall answer coverage:")
print(df["has_answer"].value_counts(normalize=True))

print()
print("Answer coverage by source folder (lowest first — expect the 3 MedlinePlus-copyright folders near the bottom):")
df.groupby("source_folder")["has_answer"].mean().sort_values()

Overall answer coverage:
has_answer
False    0.65416
True     0.34584
Name: proportion, dtype: float64

Answer coverage by source folder (lowest first — expect the 3 MedlinePlus-copyright folders near the bottom):


,has_answer
source_folder,
10_MPlus_ADAM_QA,0.000000
11_MPlusDrugs_QA,0.000000
12_MPlusHerbsSupplements_QA,0.000000
2_GARD_QA,0.999073
1_CancerGov_QA,1.000000
3_GHR_QA,1.000000
4_MPlus_Health_Topics_QA,1.000000
5_NIDDK_QA,1.000000
6_NINDS_QA,1.000000


## 4. Question type distribution
Across all 37 documented qtypes — which are most common in this corpus.

In [ ]:
df["qtype"].value_counts()

,count
qtype,
information,9214
symptoms,4338
treatment,3906
causes,2436
outlook,2232
exams and tests,2058
when to contact a medical professional,1738
inheritance,1446
precautions,1413


## 5. Source folder sizes
How many QA pairs come from each of the 12 NIH sources.

In [ ]:
df["source_folder"].value_counts()

,count
source_folder,
10_MPlus_ADAM_QA,17348
11_MPlusDrugs_QA,12889
3_GHR_QA,5430
2_GARD_QA,5394
5_NIDDK_QA,1192
6_NINDS_QA,1088
4_MPlus_Health_Topics_QA,981
12_MPlusHerbsSupplements_QA,792
7_SeniorHealth_QA,769


## 6. Answer length (where present)

In [ ]:
answered = df[df["has_answer"]].copy()
answered["answer_word_count"] = answered["answer"].str.split().str.len()

answered["answer_word_count"].describe()

,answer_word_count
count,16407.000000
mean,201.354361
std,248.480189
min,1.000000
25%,71.000000
50%,138.000000
75%,252.000000
max,4281.000000
